# M1 — Case type

**Approach:** summary-only text classifier (contributor A).
Word + character TF-IDF into a class-balanced logistic regression. **Group-aware split** so identical summaries never straddle train/test. Normalised evaluation (`P.evaluate`) plus a confusion matrix and per-class report. Saves `models/m1_case_type.joblib`.

In [ ]:
import sys, os, warnings; warnings.filterwarnings("ignore")
_p = os.getcwd()
while _p != os.path.dirname(_p) and not os.path.exists(os.path.join(_p, "northstar_prep.py")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import numpy as np, pandas as pd
import northstar_prep as P

In [ ]:
df = P.load_clean()
FEATS, TARGET = ["case_summary"], "case_type"
labels = sorted(df[TARGET].unique())
tr, te = P.group_split(df)
pipe = P.make_clf([P.t_word(), P.t_char()]).fit(tr[FEATS], tr[TARGET])
pred = pipe.predict(te[FEATS])
base = df[TARGET].value_counts(normalize=True).iloc[0]
metrics = P.evaluate(te[TARGET], pred, labels, baseline=base)
metrics

In [ ]:
proba = pipe.predict_proba(te[FEATS])
test = te[["case_id","case_summary",TARGET]].copy().reset_index(drop=True)
test["predicted"], test["confidence"] = pred, proba.max(axis=1).round(3)
full = P.make_clf([P.t_word(), P.t_char()]).fit(df[FEATS], df[TARGET])
P.save_bundle("m1_case_type", {
    "model": full, "target": TARGET, "features": FEATS, "classes": labels, "kind": "multiclass",
    "label": "Case type", "metrics": metrics, "test": test,
    "report": P.class_report(te[TARGET], pred, labels), "confusion": P.conf_matrix(te[TARGET], pred, labels),
    "meta": {"contributor":"A", "approach":"summary-only text classifier",
             "split":"group-on-summary", "inputs":"summary only"}})